# RNAscope: LD4 and dual-channel LP

Quantify spots directly in raw Imaris IMS images with the original Big-FISH
detection and dense-region decomposition. External 3D label masks must match
the original Z,Y,X image geometry. Mask labels 1–4 are sLNv cells; label 5
is the 5th sLNv. Each sample contributes the mean of labels 1–4 and one value
for label 5. The original absent-label-as-zero rule is preserved.

Use Python 3.11 and `requirements.txt`. Images and manual masks are supplied
separately; `example_data/` is currently a placeholder. No images are needed
to edit the configuration. Each real run exports a separate report folder.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = None  # Set explicitly if the kernel starts outside this repository.
if PROJECT_ROOT is None:
    candidates = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([parent, parent / 'RNAscope analysis'])
    PROJECT_ROOT = next((p for p in candidates if (p / 'script' / 'rnascope_workflow.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Set PROJECT_ROOT to this analysis folder, then rerun this cell.')
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
script_dir = PROJECT_ROOT / 'script'
loaded = sys.modules.get('rnascope_quant')
expected = script_dir / 'rnascope_quant'
if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(script_dir):
    raise RuntimeError('Restart the kernel: a different copy of rnascope_quant is already imported.')
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))
print('Analysis folder:', PROJECT_ROOT)


In [ ]:
from rnascope_workflow import (
    load_experiment, plan_experiment, run_quantification,
    preprocess_dataset, export_spot_markers,
)

EXPERIMENT = 'LD4'  # 'LD4' or 'LP_dual'; one configuration per run.
DATA_ROOT = PROJECT_ROOT / 'example_data'  # Change to your raw-data parent directory.
CONFIG_FILE = PROJECT_ROOT / 'config' / f'{EXPERIMENT}.json'
config = load_experiment(CONFIG_FILE, data_root=DATA_ROOT)
print('Experiment:', config['experiment'])
print('Detection parameters:', config['detection'])


## 1. Inspect the image/mask pairing

LD4 reads raw channel 0 and `_CH1_mask.nrrd` labels. LP reads raw channels
0 and 1; within each dataset it uses `_CH3_mask.nrrd` when present, otherwise
`_CH2_mask.nrrd`, preserving the original suffix priority. A mask is paired
to the IMS with the same complete basename. Dataset IDs distinguish samples
with identical basenames in different batches. Relative dataset paths are
resolved under `DATA_ROOT`; individual absolute directories are also accepted.

In [ ]:
plan = plan_experiment(config)
display(plan)


## 2. Run quantification

LD4 uses a fixed threshold of 100. LP uses CH0=100 and CH1=30. Both keep
300-nm spot radii and 500×96×96-nm Z,Y,X voxels from the original notebook.
Check these against the acquisition before analyzing other images.

Matching checkpoints include input file signatures, parameters, software
and code versions. Changing a threshold or mask invalidates the old result;
old filename-only checkpoints are not read. `save_spots` saves actual raw
and final detection coordinates for later QC, without re-detection.

In [ ]:
RUN_QUANTIFICATION = False
REUSE_MATCHING_CACHE = True
result = None
if RUN_QUANTIFICATION:
    result = run_quantification(config, reuse_cache=REUSE_MATCHING_CACHE)
    display(result['raw_data'].head())
    print('Workbook:', result['workbook'])
    print('Cache hits:', result['cache_hits'])
    print('Failed samples:', len(result['failures']))
    display(result['failures'])


## 3. Statistics and documented exclusions

Exclusions are configured separately for each experiment as dictionaries
with `pattern` (a case-insensitive regular expression) and `reason`. The
default list is empty: no exclusions have been inferred from other analyses.
Every removed sample and matching reason is exported. Original counts remain
in `Raw_Data`; filtered data and statistics are saved separately.

`random_seed=null` preserves the original unspecified random state; the
state before each new detection is recorded. An integer enables stable
per-sample seeds for new runs. Historic results without a recorded seed
cannot be assumed to reproduce exactly after dense-region decomposition.

In [ ]:
if result is not None:
    if config['mode'] == 'single':
        display(result['statistics'])
    else:
        display(result['statistics'][0])
        display(result['statistics'][1])
    display(result['removed_samples'])


## 4. Optional structural-channel export for manual annotation

This exports a log-transformed, percentile-normalized helper channel as
NRRD. Choose the structural channel explicitly. Create the labeled cell
masks externally, then save them to the dataset's configured mask directory.
Normalized helper images are not used as spot-detection inputs.

In [ ]:
RUN_PREPROCESSING = False
ANNOTATION_DATASET_ID = config['datasets'][0]['id']
STRUCTURAL_CHANNEL = 1 if config['mode'] == 'single' else 2
if RUN_PREPROCESSING:
    output = preprocess_dataset(config, ANNOTATION_DATASET_ID, channel_index=STRUCTURAL_CHANNEL)
    print('Helper channel output:', output)


## 5. Optional 3D markers from the saved detection

Select a row of `spot_index` to choose a sample/channel. Archives contain
zero-based Z,Y,X coordinates and voxel size. The marker stack is a QC aid,
not a cell segmentation. Keep the cache directory with the run reports:
`Spot_Files` references its archives by relative paths.

In [ ]:
RUN_MARKER_EXPORT = False
if result is not None:
    display(result['spot_index'].head())
if RUN_MARKER_EXPORT:
    if result is None or result['spot_index'].empty:
        raise RuntimeError('Run quantification with save_spots=true first.')
    selected = result['spot_index'].iloc[0]
    spot_file = (result['output_dir'] / selected['spot_file']).resolve()
    markers = export_spot_markers(spot_file, result['output_dir'] / 'selected_sample_markers.tif')
